<a href="https://colab.research.google.com/github/NadiiaStuzhuk/DI-Bootcamp-Stage1/blob/main/Mini_project_Brazilian_e_commerce_Student_NS_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1 & 2. Load Data, Create SQLite Database, and Export DataFrames
# Below is the complete setup script using pandas and sqlalchemy to load all 9 Olist CSV files, export them into an in-memory SQLite database, and verify the table load.

import numpy as np
import pandas as pd
import sqlite3
from sqlalchemy import create_engine

# 1. Load datasets (Adjust path to './' if files are in the current working directory)
DATA_DIR = '/kaggle/input/brazilian-ecommerce/'  # Or './' in local/Colab

df_olist_customers = pd.read_csv(f'{DATA_DIR}olist_customers_dataset.csv')
df_olist_sellers = pd.read_csv(f'{DATA_DIR}olist_sellers_dataset.csv')
df_olist_order_reviews = pd.read_csv(f'{DATA_DIR}olist_order_reviews_dataset.csv')
df_olist_order_items = pd.read_csv(f'{DATA_DIR}olist_order_items_dataset.csv')
df_olist_products = pd.read_csv(f'{DATA_DIR}olist_products_dataset.csv')
df_olist_geolocation = pd.read_csv(f'{DATA_DIR}olist_geolocation_dataset.csv')
df_product_category_name_translation = pd.read_csv(f'{DATA_DIR}product_category_name_translation.csv')
df_olist_orders = pd.read_csv(f'{DATA_DIR}olist_orders_dataset.csv')
df_olist_order_payments = pd.read_csv(f'{DATA_DIR}olist_order_payments_dataset.csv')

# 2. Create SQLite database engine and export each DataFrame as a SQL table
engine = create_engine('sqlite://', echo=False)

df_olist_customers.to_sql("olist_customers", con=engine, if_exists="replace")
df_olist_sellers.to_sql("olist_sellers", con=engine, if_exists="replace")
df_olist_order_reviews.to_sql("olist_order_reviews", con=engine, if_exists="replace")
df_olist_order_items.to_sql("olist_order_items", con=engine, if_exists="replace")
df_olist_products.to_sql("olist_products_dataset", con=engine, if_exists="replace")
df_olist_geolocation.to_sql("olist_geolocation", con=engine, if_exists="replace")
df_product_category_name_translation.to_sql("product_category_name_translation", con=engine, if_exists="replace")
df_olist_orders.to_sql("olist_orders", con=engine, if_exists="replace")
df_olist_order_payments.to_sql("olist_order_payments", con=engine, if_exists="replace")

# Test connection by selecting the first 5 rows of olist_customers
sql = '''
SELECT *
FROM olist_customers
LIMIT 5;
'''
df_sql = pd.read_sql_query(sql, con=engine)
display(df_sql)


# 3. Query 1: Count and Percentage of Orders Purchased in Jan 2018 with a 5-Star Review Score
# This query joins olist_orders and olist_order_reviews, filters for orders placed in January 2018 (2018-01), counts how many received a review_score = 5, and calculates their percentage relative to all reviewed orders purchased in January 2018.
sql_q1 = '''
SELECT
    COUNT(DISTINCT CASE WHEN r.review_score = 5 THEN o.order_id END) AS five_star_orders_jan_2018,
    COUNT(DISTINCT o.order_id) AS total_orders_jan_2018,
    ROUND(
        COUNT(DISTINCT CASE WHEN r.review_score = 5 THEN o.order_id END) * 100.0 /
        COUNT(DISTINCT o.order_id), 2
    ) AS five_star_percentage
FROM olist_orders o
JOIN olist_order_reviews r
    ON o.order_id = r.order_id
WHERE STRFTIME('%Y-%m', o.order_purchase_timestamp) = '2018-01';
'''

df_q1 = pd.read_sql_query(sql_q1, con=engine)
display(df_q1)

# 4. Query 2: Customer Purchase Trend Year-on-Year
# This query extracts the purchase year and month, aggregating total orders, unique customers (customer_unique_id), and year-over-year order growth across 2016, 2017, and 2018.

sql_q2 = '''
WITH yearly_stats AS (
    SELECT
        STRFTIME('%Y', o.order_purchase_timestamp) AS purchase_year,
        COUNT(DISTINCT o.order_id) AS total_orders,
        COUNT(DISTINCT c.customer_unique_id) AS unique_customers
    FROM olist_orders o
    JOIN olist_customers c
        ON o.customer_id = c.customer_id
    GROUP BY purchase_year
)
SELECT
    purchase_year,
    total_orders,
    unique_customers,
    LAG(total_orders) OVER (ORDER BY purchase_year) AS prev_year_orders,
    ROUND(
        (total_orders - LAG(total_orders) OVER (ORDER BY purchase_year)) * 100.0 /
        LAG(total_orders) OVER (ORDER BY purchase_year), 2
    ) AS yoy_order_growth_pct
FROM yearly_stats
ORDER BY purchase_year;
'''

df_q2 = pd.read_sql_query(sql_q2, con=engine)
display(df_q2)


# Verified Results from Uploaded olist_orders_dataset.csv & olist_customers_dataset.csv:
# purchase_year,total_orders,unique_customers,prev_year_orders,yoy_order_growth_pct
# 2016,329,326,NULL,NULL
# 2017,"45,101","43,713",329,"13,608.51%"
# 2018,"54,011","52,749","45,101",19.76%

# 5. Query 3: Average Order Values (AOV) of Customers
# This query computes the Average Order Value (AOV) per customer as well as the overall platform AOV using olist_order_payments (or price + freight_value from olist_order_items).

sql_q3 = '''
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        c.customer_city,
        c.customer_state,
        SUM(p.payment_value) AS order_value
    FROM olist_orders o
    JOIN olist_customers c
        ON o.customer_id = c.customer_id
    JOIN olist_order_payments p
        ON o.order_id = p.order_id
    WHERE o.order_status != 'canceled'
    GROUP BY o.order_id, c.customer_unique_id, c.customer_city, c.customer_state
)
SELECT
    customer_unique_id,
    customer_city,
    customer_state,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(order_value), 2) AS total_spent,
    ROUND(AVG(order_value), 2) AS avg_order_value
FROM order_totals
GROUP BY customer_unique_id, customer_city, customer_state
ORDER BY avg_order_value DESC
LIMIT 15;
'''

df_q3 = pd.read_sql_query(sql_q3, con=engine)
display(df_q3)

# 6. Query 4: Top 5 Cities with Highest Revenue from 2016 to 2018
# This query joins olist_orders, olist_customers, and olist_order_payments to rank the top 5 Brazilian cities generating the highest total revenue between 2016 and 2018.

sql_q4 = '''
SELECT
    c.customer_city,
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(p.payment_value), 2) AS total_revenue
FROM olist_orders o
JOIN olist_customers c
    ON o.customer_id = c.customer_id
JOIN olist_order_payments p
    ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
  AND STRFTIME('%Y', o.order_purchase_timestamp) BETWEEN '2016' AND '2018'
GROUP BY c.customer_city, c.customer_state
ORDER BY total_revenue DESC
LIMIT 5;
'''

df_q4 = pd.read_sql_query(sql_q4, con=engine)
display(df_q4)

# 7. Query 5: State-Wise Revenue Table Between 2016 and 2018
# This query builds a complete state-level revenue summary (including year-by-year pivot columns for 2016, 2017, and 2018 alongside total revenue and average order value).

sql_q5 = '''
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(CASE WHEN STRFTIME('%Y', o.order_purchase_timestamp) = '2016' THEN p.payment_value ELSE 0 END), 2) AS rev_2016,
    ROUND(SUM(CASE WHEN STRFTIME('%Y', o.order_purchase_timestamp) = '2017' THEN p.payment_value ELSE 0 END), 2) AS rev_2017,
    ROUND(SUM(CASE WHEN STRFTIME('%Y', o.order_purchase_timestamp) = '2018' THEN p.payment_value ELSE 0 END), 2) AS rev_2018,
    ROUND(SUM(p.payment_value), 2) AS total_revenue_2016_2018,
    ROUND(SUM(p.payment_value) / COUNT(DISTINCT o.order_id), 2) AS state_aov
FROM olist_orders o
JOIN olist_customers c
    ON o.customer_id = c.customer_id
JOIN olist_order_payments p
    ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
  AND STRFTIME('%Y', o.order_purchase_timestamp) BETWEEN '2016' AND '2018'
GROUP BY c.customer_state
ORDER BY total_revenue_2016_2018 DESC;
'''

df_q5 = pd.read_sql_query(sql_q5, con=engine)
display(df_q5)


# 8. Query 6: Top Successful Sellers (Goods Sold, Revenue, Customer Count, and 5-Star Ratings)
# This query evaluates seller performance across all four requested dimensions: total items sold, total revenue generated (price), distinct customers served, and total 5-star reviews earned.

sql_q6 = '''
SELECT
    s.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(oi.order_item_id) AS total_goods_sold,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    COUNT(DISTINCT o.customer_id) AS unique_customer_count,
    SUM(CASE WHEN r.review_score = 5 THEN 1 ELSE 0 END) AS five_star_ratings_count,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM olist_sellers s
JOIN olist_order_items oi
    ON s.seller_id = oi.seller_id
JOIN olist_orders o
    ON oi.order_id = o.order_id
LEFT JOIN olist_order_reviews r
    ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY s.seller_id, s.seller_city, s.seller_state
ORDER BY total_revenue DESC, total_goods_sold DESC, five_star_ratings_count DESC
LIMIT 10;
'''

df_q6 = pd.read_sql_query(sql_q6, con=engine)
display(df_q6)

# 9. Query 7: Delivery Success Rate Across States
# This query calculates both the Overall Delivery Completion Rate (order_status = 'delivered') and the On-Time Delivery Rate (order_delivered_customer_date <= order_estimated_delivery_date) across Brazilian states.

sql_q7 = '''
SELECT
    c.customer_state,
    COUNT(o.order_id) AS total_orders,
    SUM(CASE WHEN o.order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    ROUND(
        SUM(CASE WHEN o.order_status = 'delivered' THEN 1.0 ELSE 0.0 END) * 100.0 /
        COUNT(o.order_id), 2
    ) AS delivery_success_rate_pct,
    ROUND(
        SUM(CASE WHEN o.order_status = 'delivered'
                  AND o.order_delivered_customer_date <= o.order_estimated_delivery_date
                 THEN 1.0 ELSE 0.0 END) * 100.0 /
        NULLIF(SUM(CASE WHEN o.order_status = 'delivered' THEN 1 ELSE 0 END), 0), 2
    ) AS on_time_delivery_rate_pct
FROM olist_orders o
JOIN olist_customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total_orders DESC;
'''

df_q7 = pd.read_sql_query(sql_q7, con=engine)
display(df_q7)

# 10. Query 8: Preferred Form of Payment for Different Categories
# This query uses a Common Table Expression (CTE) and ROW_NUMBER() window function to find the #1 most frequently used payment_type (e.g., credit_card, boleto, voucher, debit_card) for each English product category.

sql_q8 = '''
WITH category_payment_counts AS (
    SELECT
        COALESCE(t.product_category_name_english, p.product_category_name, 'Unknown') AS category_name,
        pay.payment_type,
        COUNT(DISTINCT o.order_id) AS order_count,
        ROUND(SUM(pay.payment_value), 2) AS total_payment_value,
        ROW_NUMBER() OVER (
            PARTITION BY COALESCE(t.product_category_name_english, p.product_category_name, 'Unknown')
            ORDER BY COUNT(DISTINCT o.order_id) DESC
        ) AS rn
    FROM olist_order_items oi
    JOIN olist_orders o
        ON oi.order_id = o.order_id
    JOIN olist_products_dataset p
        ON oi.product_id = p.product_id
    LEFT JOIN product_category_name_translation t
        ON p.product_category_name = t.product_category_name
    JOIN olist_order_payments pay
        ON o.order_id = pay.order_id
    GROUP BY category_name, pay.payment_type
)
SELECT
    category_name,
    payment_type AS preferred_payment_method,
    order_count,
    total_payment_value
FROM category_payment_counts
WHERE rn = 1
ORDER BY order_count DESC
LIMIT 20;
'''

df_q8 = pd.read_sql_query(sql_q8, con=engine)
display(df_q8)

# 11. Query 9: Distance Between Cities (Customer City vs. Seller City)To compute the geographical distance between the customer's city and the seller's city in SQLite:First, aggregate olist_geolocation by geolocation_zip_code_prefix (using AVG(geolocation_lat) and AVG(geolocation_lng)) to prevent duplicate zip-code rows from multiplying joins.Join customer and seller coordinates to each order and apply the Haversine / Euclidean degree-to-kilometer formula ($1^\circ \approx 111.12\text{ km}$):$$d \approx 111.12 \times \sqrt{(\text{lat}_c - \text{lat}_s)^2 + \left((\text{lng}_c - \text{lng}_s) \times \cos\left(\frac{\text{lat}_c \cdot \pi}{180}\right)\right)^2}$$

sql_q9 = '''
WITH zip_coords AS (
    SELECT
        geolocation_zip_code_prefix AS zip_prefix,
        AVG(geolocation_lat) AS lat,
        AVG(geolocation_lng) AS lng
    FROM olist_geolocation
    GROUP BY geolocation_zip_code_prefix
),
order_distances AS (
    SELECT
        o.order_id,
        c.customer_city,
        c.customer_state,
        s.seller_city,
        s.seller_state,
        -- Approximate distance in kilometers using Euclidean spherical projection
        111.12 * SQRT(
            POWER(zc.lat - zs.lat, 2) +
            POWER((zc.lng - zs.lng) * COS(zc.lat * 3.14159265 / 180.0), 2)
        ) AS distance_km
    FROM olist_orders o
    JOIN olist_customers c
        ON o.customer_id = c.customer_id
    JOIN olist_order_items oi
        ON o.order_id = oi.order_id
    JOIN olist_sellers s
        ON oi.seller_id = s.seller_id
    JOIN zip_coords zc
        ON c.customer_zip_code_prefix = zc.zip_prefix
    JOIN zip_coords zs
        ON s.seller_zip_code_prefix = zs.zip_prefix
    WHERE c.customer_city != s.seller_city
)
SELECT
    customer_city,
    customer_state,
    seller_city,
    seller_state,
    COUNT(DISTINCT order_id) AS total_orders_between_cities,
    ROUND(AVG(distance_km), 2) AS avg_distance_km
FROM order_distances
GROUP BY customer_city, customer_state, seller_city, seller_state
ORDER BY total_orders_between_cities DESC
LIMIT 15;
'''

df_q9 = pd.read_sql_query(sql_q9, con=engine)
display(df_q9)

# Note for older SQLite builds without built-in math functions (SQRT/COS): If your local SQLite binary has SQLITE_ENABLE_MATH_FUNCTIONS disabled, register Python's math functions on the raw connection before running sql_q9:
import math
raw_conn = engine.raw_connection()
raw_conn.create_function("SQRT", 1, math.sqrt)
raw_conn.create_function("COS", 1, math.cos)
raw_conn.create_function("POWER", 2, math.pow)
df_q9 = pd.read_sql_query(sql_q9, con=raw_conn)